In [ ]:
import os
import sys
import yaml
sys.path.append(os.path.abspath('../src'))
import matplotlib.cm as cm

from utils import *

# Load config (relative to notebooks/)
with open('../config.yml', 'r') as f:
    config = yaml.safe_load(f)

In [ ]:
train_dir = config['paths']['train_dir']
val_dir = config['paths']['val_dir']
test_dir = config['paths']['test_dir']

In [ ]:
# ── 1. Load Pre-trained Models ────────────────────────────────────────────────
# Load the trained models. Note that each model expects a different 
# input spatial resolution (256x256 and 300x300).
model_256 = keras.saving.load_model("../models/resnet50/resnet50_v3_best.keras")
model_300 = keras.saving.load_model("../models/EfficientNetV2S/effnetv2s_v1_best.keras")

# ── 2. Prepare Test Datasets ──────────────────────────────────────────────────
test_ds_256 = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(256, 256),
    batch_size=16,
    shuffle=False,
    label_mode="int"
)

# Extract class names BEFORE applying .prefetch()
class_names = test_ds_256.class_names

# Apply prefetch for performance optimization
test_ds_256 = test_ds_256.prefetch(tf.data.AUTOTUNE)

test_ds_300 = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(300, 300),
    batch_size=16,
    shuffle=False,
    label_mode="int"
).prefetch(tf.data.AUTOTUNE)

# ── 3. Extract Test Probabilities ─────────────────────────────────────────────
probs_test_256 = model_256.predict(test_ds_256)
probs_test_300 = model_300.predict(test_ds_300)

# ── 4. Extract Ground Truth Labels ────────────────────────────────────────────
y_test_true = np.concatenate([y for _, y in test_ds_256], axis=0)

# ── 5. Prepare Validation Datasets ────────────────────────────────────────────
val_ds_256 = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=(256, 256),
    batch_size=16,
    shuffle=False,
    label_mode="int"
).prefetch(tf.data.AUTOTUNE)

val_ds_300 = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=(300, 300),
    batch_size=16,
    shuffle=False,
    label_mode="int"
).prefetch(tf.data.AUTOTUNE)

probs_val_256 = model_256.predict(val_ds_256)
probs_val_300 = model_300.predict(val_ds_300)
y_val_true    = np.concatenate([y for _, y in val_ds_256], axis=0)

# ── 6. Optimize Ensemble Weights via Grid Search ──────────────────────────────
best_f1, best_w = 0, 0
for w in np.arange(0.0, 1.05, 0.05):
    probs = w * probs_val_256 + (1 - w) * probs_val_300
    preds = np.argmax(probs, axis=1)
    f1 = f1_score(y_val_true, preds, average="macro")
    if f1 > best_f1:
        best_f1, best_w = f1, w

print(f"Optimal Weight (Model 256): {best_w:.2f} | Optimal Weight (Model 300): {1-best_w:.2f}")
print(f"Validation Maximum Macro F1: {best_f1:.4f}")

# ── 7. Evaluate Ensemble Performance on Test Set ──────────────────────────────
ensemble_probs_test = best_w * probs_test_256 + (1 - best_w) * probs_test_300
ensemble_preds_test = np.argmax(ensemble_probs_test, axis=1)

metrics_ensemble = evaluate_from_predictions(
    y_true=y_test_true, 
    y_pred=ensemble_preds_test, 
    y_pred_probs=ensemble_probs_test, 
    class_names=class_names, 
    model_name="Ensemble Base (ResNet50 + EffNetV2S)",
    test_loss=None
)

In [ ]:
# ── 8. Ensemble with Test-Time Augmentation (TTA) ─────────────────────────────
# This cell performs TTA manually to bypass the bug in utils.py caused by 
# integer labels, while maintaining optimal memory usage for the GTX 1650 - the GPU used to run this cell.

n_augmentations = 10
print(f"--- Starting TTA Ensemble with {n_augmentations} augmentations ---")

# Define a simple augmentation pipeline to be applied during inference
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

# Initialize accumulators for the probabilities
# We use the existing probability shapes from the non-TTA predictions
probs_tta_256 = np.zeros_like(probs_test_256)
probs_tta_300 = np.zeros_like(probs_test_300)

for i in range(n_augmentations):
    print(f"  Processing Augmentation {i+1}/{n_augmentations}...")
    
    # Map the augmentation to the datasets (training=True forces augmentation at inference)
    aug_ds_256 = test_ds_256.map(lambda x, y: (data_augmentation(x, training=True), y))
    aug_ds_300 = test_ds_300.map(lambda x, y: (data_augmentation(x, training=True), y))
    
    # Predict and accumulate results
    probs_tta_256 += model_256.predict(aug_ds_256, verbose=0)
    probs_tta_300 += model_300.predict(aug_ds_300, verbose=0)

# Compute the mean probabilities across all TTA iterations
probs_tta_256 /= n_augmentations
probs_tta_300 /= n_augmentations

# ── Weighted TTA Fusion ───────────────────────────────────────────────────────
# Apply the optimal weights derived from the previous validation grid search.
print(f"\n--- Finalizing TTA Ensemble (M256 weight: {best_w:.2f}) ---")

ensemble_probs_tta = best_w * probs_tta_256 + (1 - best_w) * probs_tta_300
ensemble_preds_tta = np.argmax(ensemble_probs_tta, axis=1)

# Generate the final classification report and confusion matrix using y_test_true
metrics_ensemble_tta = evaluate_from_predictions(
    y_true=y_test_true, 
    y_pred=ensemble_preds_tta, 
    y_pred_probs=ensemble_probs_tta, 
    class_names=class_names, 
    model_name="Ensemble (M256 + M300) w/ TTA"
)